In [0]:
%run ../utils/init.py

In [0]:
files = dbutils.fs.ls(f"{RAW}/dvf/annee=2020/")
for f in files:
    print(f.name, f.size)

In [0]:
IDF_DEPTS = ['75', '77', '78', '91', '92', '93', '94', '95']

df_raw = spark.read.parquet(f"{RAW}/dvf/") \
    .filter(F.col("code_departement").isin(IDF_DEPTS))

print(f"Lignes IDF brutes : {df_raw.count()}")

In [0]:
from pyspark.sql.functions import current_timestamp

df_bronze = (df_raw
    .filter(F.col("nature_mutation") == "Vente")
    .filter(F.col("valeur_fonciere").between(1_000, 50_000_000))
    .filter(F.col("valeur_fonciere").isNotNull())
    .withColumn("prix_m2", 
        F.when(F.col("surface_reelle_bati") > 0,
               F.col("valeur_fonciere") / F.col("surface_reelle_bati"))
    )
    .withColumn("ingestion_timestamp", current_timestamp())
)

print(f"Lignes après filtres : {df_bronze.count()}")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("annee", "code_departement") \
    .save(f"{BRONZE}/dvf_transactions/")

print("✅ Bronze DVF écrit")